# RealVisXL V4.0 Uncensored Studio (Original Model Restored)
Restored RealVisXL V4.0 with safety checker disabled and memory optimization for T4 GPU.

In [ ]:
!pip install -q diffusers transformers accelerate torch torchvision gradio safetensors

import torch
from diffusers import AutoPipelineForImage2Image
import gradio as gr

has_cuda = torch.cuda.is_available()
device = "cuda" if has_cuda else "cpu"
dtype = torch.float16 if has_cuda else torch.float32

print("Loading RealVisXL V4.0 (Original Model)...")
pipe_i2i = AutoPipelineForImage2Image.from_pretrained(
    "SG161222/RealVisXL_V4.0",
    torch_dtype=dtype,
    use_safetensors=True
)

if hasattr(pipe_i2i, "safety_checker"):
    pipe_i2i.safety_checker = None
if hasattr(pipe_i2i, "requires_safety_checker"):
    pipe_i2i.requires_safety_checker = False

if has_cuda:
    pipe_i2i.enable_sequential_cpu_offload()
else:
    pipe_i2i.to("cpu")

def edit(image, prompt, negative_prompt, strength, steps, cfg):
    if image is None:
        return None
    
    image = image.convert("RGB")
    image.thumbnail((1024, 1024))
    
    full_prompt = prompt + ", raw photo, highly detailed, photorealistic"
    
    img = pipe_i2i(
        prompt=full_prompt,
        negative_prompt=negative_prompt,
        image=image,
        strength=float(strength),
        num_inference_steps=int(steps),
        guidance_scale=float(cfg)
    ).images[0]
    return img

demo = gr.Interface(
    fn=edit,
    inputs=[
        gr.Image(type="pil", label="Source Image"),
        gr.Textbox(label="Prompt", value="photorealistic, high quality"),
        gr.Textbox(label="Negative Prompt", value="blurry, low quality, deformed"),
        gr.Slider(0.1, 0.6, value=0.25, label="Strength (0.25 preserves identity)"),
        gr.Slider(20, 50, value=30, step=1, label="Steps"),
        gr.Slider(4.0, 10.0, value=7.0, step=0.5, label="CFG Scale")
    ],
    outputs=gr.Image(label="Result"),
    title="RealVisXL V4.0 Studio"
)
demo.launch(share=True, debug=True)